# 02 · Vision Transformers

> **Source notes:** `VisionTransformers.md`

How does a ViT turn a `(3, 224, 224)` image tensor into a sequence a transformer can process?

This notebook builds it from scratch:
- **Patch extraction** — split an image into 16×16 patches manually
- **Patch embedding** — project each patch to a 768-dim vector
- **Positional embeddings** — add learned position information
- **Attention visualisation** — show which patches attend to which
- **PixelSmith v1** — run a real ViT-B/16 on our synthetic image

**All local. No GPU needed.** Runtime: < 1 minute.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `run()` to produce the result
#
# Hint:
#    # implement using the APIs described above

## 1 · Manual Patch Extraction

A ViT splits the image into non-overlapping $P \times P$ patches.

For a 224×224 image with $P = 16$: $N = (224/16)^2 = 196$ patches.

In [ ]:
# TODO: Implement this cell
#  (Manual patch extraction using unfold)
#
# Steps:
# 1. Process data
# 2. Compute `img_np` using `linspace()`
# 3. Compute `img` using `tensor()`
# 4. Manual patch extraction using unfold
# 5. Compute `patches` using `unfold()`
# 6. Compute `N` using `to()`
#
# Hint:
#    img_np = np.zeros(???)
#    img = torch.from_numpy(???)
#    patches = img.unfold(???)
#    patches_flat = patches.contiguous(???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `suptitle()`
#
# Hint:
#    axes = plt.subplots(???)

## 2 · Building a Minimal ViT Patch Embedding Layer

The patch embedding is a single `Conv2d(in_channels=C, out_channels=d, kernel_size=P, stride=P)`. This is mathematically equivalent to the linear projection — it just uses optimised GPU kernels.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `PatchEmbedding()` to produce the result
# 2. Define helper function
# 3. Call `Conv2d()` to produce the result
# 4. Define helper function
# 5. Compute `d_model` using `PatchEmbedding()`
# 6. Compute `patch_embeddings` using `patch_embed()`
#
# Hint:
#    patch_embed = PatchEmbedding(img_size=???, patch_size=???)
#    projection = nn.Conv2d(???)
#    x = self.projection(???)
#    x = x.flatten(???)

## 3 · CLS Token and Positional Embeddings

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `N_PATCHES`
# 2. Compute `cls_token` using `Parameter()`
# 3. Compute `pos_embed` using `Parameter()`
# 4. Compute `cls_expanded` using `expand()`
# 5. Compute `seq_with_pos`
# 6. Call `token()` to produce the result
#
# Hint:
#    cls_token = nn.Parameter(???)
#    pos_embed = nn.Parameter(???)
#    cls_expanded = cls_token.expand(???)
#    seq = torch.cat(???)

## 4 · Self-Attention Over Patches — Minimal Implementation

Standard scaled dot-product attention, now applied to patch embeddings instead of text tokens.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """
    TODO #6: Implement `scaled_dot_product_attention()`.

    Steps:
    1. Define helper function `scaled_dot_product_attention()`
    2. Compute `x_small` using `randn()`
    3. Compute `Wq` using `Linear()`
    4. Compute `Q` using `unsqueeze()`
    5. Call `squeeze()` to produce the result
    6. Plot results -- call `subplots()`
    7. Call `detach()` to produce the result

    Hint:
    Q = Wq(???)
    K = Wk(???)
    scores = torch.matmul(???)
    weights = F.softmax(???)

    Returns: torch.matmul(weights, V), weights
    """
    raise NotImplementedError("TODO: implement scaled_dot_product_attention()")

## 5 · PixelSmith v1 — Real ViT-B/16 via torchvision

Run the full pretrained ViT-B/16 on our synthetic image to get a real 768-dim embedding.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `vit` using `16()`
# 3. Compute `total_params` using `numel()`
# 4. Compute `transform` using `Compose()`
# 5. Process data
# 6. Compute `url` using `urlopen()`
# 7. Compute `img_tensor` using `transform()`
#
# Hint:
#    vit = models.vit_b_16(???)
#    transform = T.Compose(???)
#    img_real = Image.open(???)
#    img_real = Image.fromarray(???)

In [ ]:
# TODO: Implement this cell
#  (Extract the CLS token embedding (before the classification head))
#
# Steps:
# 1. Extract the CLS token embedding (before the classification head)
# 2. Call `item()` to produce the result
#
# Hint:
#    features = vit._process_input(???)
#    cls = vit.class_token.expand(???)
#    seq = torch.cat(???)
#    encoded = vit.encoder.layers(???)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `works()` to produce the result
# 2. Compute `imagenet_labels` using `names()`
#
# Hint:
#    top5_idx = logits.topk(???)
#    label = imagenet_labels.get(???)

## 6 · Positional Embedding Similarity

ViT's positional embeddings learn geometric structure from data. The cosine similarity between adjacent positions should be higher than between distant positions.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `no_grad()` to produce the result
# 2. Call `numpy()` to produce the result
# 3. Plot results -- call `heatmap()`
# 4. Compute `reference_patches`
# 5. Plot results -- call `reshape()`
# 6. Plot results -- call `suptitle()`
# 7. Process data
#
# Hint:
#    pos_norm = F.normalize(???)
#    axes = plt.subplots(???)
#    im = ax.imshow(???)

## 7 · Summary — PixelSmith v1

```
┌──────────────────────────────────────────────────────────────────────────────┐
│ PixelSmith v1 — Visual Frontend │
│ │
│ (3, 224, 224) image tensor (from Ch.1) │
│ │ │
│ ▼ Conv2d(P=16, stride=16) │
│ (196, 768) patch embeddings │
│ │ │
│ ▼ Prepend [CLS], add pos embeddings │
│ (197, 768) sequence │
│ │ │
│ ▼ 12 × Transformer encoder layers (MSA + MLP) │
│ (197, 768) encoded sequence │
│ │ │
│ ▼ Take CLS token (position 0) │
│ (768,) image embedding ← ready for CLIP alignment (Ch.3) │
└──────────────────────────────────────────────────────────────────────────────┘
```

**Next:** [CLIP.md](../ch03_clip/CLIP.md) — train a text encoder alongside this ViT with contrastive loss so that image and text embeddings live in the same space.